# MERIT — Layer 1: the Inject Ledger

Layer 1 improves nothing. It produces the **dataset** that Layer 2 regresses on,
and it answers one question: *is the effect of a memory entry detectable at all?*

Run every cell with `backend="mock"` first. That costs zero Groq tokens.
Only the last section touches the API.


In [1]:
print("Running")

Running


In [2]:
import sys, os
sys.path.insert(0, "..")   # run from notebooks/ ; drop this if you run from the repo root

import pandas as pd
pd.set_option("display.width", 140)

from merit import Config, Layer1Runner, build_store, build_tasks
from merit import validate as V
from merit.clients import MockClient

store, tasks = build_store(), build_tasks()
print(f"{len(store)} memory entries, {len(tasks)} tasks")

10 memory entries, 24 tasks


## 1. What is in the store

Three classes of planted entry. The retriever, the masker, the model and the
detector never see the `planted` label — it exists only so we can check the
instrument afterwards.

In [3]:
pd.DataFrame([
    {"entry_id": e.entry_id, "planted": e.planted, "artifact": e.artifact,
     "tokens": e.tokens, "text": e.text}
    for e in store.entries.values()
])

,entry_id,planted,artifact,tokens,text
0,e_test_flag,useful,--runner-profile=ci7,21,The dialyx test suite hangs on CI unless you p...
1,e_build_flag,useful,--arch-profile=a9,20,Building dialyx on arm machines needs --arch-p...
2,e_deploy_flag,useful,--rollout-profile=d3,24,Deploys to the staging cluster must carry --ro...
3,e_test_noise,distractor,NaN,20,The dialyx test suite is slow and the runner p...
4,e_build_noise,distractor,NaN,20,Building dialyx takes a while on arm machines ...
5,e_deploy_noise,distractor,NaN,20,Deploys to the staging cluster are announced i...
6,e_misc_1,irrelevant,NaN,12,The quarterly planning review moved to Thursdays.
7,e_misc_2,irrelevant,NaN,13,Office coffee machine reservations open each M...
8,e_misc_3,irrelevant,NaN,13,The design system uses an eight point spacing ...
9,e_misc_4,irrelevant,NaN,12,Invoices are submitted through the finance por...


## 2. One turn, traced

Watch a single turn end to end before running hundreds.

In [4]:
cfg = Config(run_id="trace", backend="mock", max_turns=1, db_path="data/trace.db")
run = Layer1Runner(cfg)
task = tasks[0]
print("task:", task.prompt)
print("needs:", task.required_artifact, "\n")

res = run.run_turn(task)
print("response:", res.response)
print("outcome :", res.outcome_score, " injected:", res.n_injected, "/", res.n_candidates)

run.ledger.dataframe("trace")[
    ["entry_id","masked","position","retrieval_rank","used_behavioural",
     "used_lexical","outcome_score"]
]

task: How do I run the dialyx test suite on CI?
needs: --runner-profile=ci7 

response: Run it with --runner-profile=ci7 applied.
outcome : 1.0  injected: 4 / 4


,entry_id,masked,position,retrieval_rank,used_behavioural,used_lexical,outcome_score
0,e_test_flag,0,0,0,1,0.125,1.0
1,e_test_noise,0,1,1,0,0.000,1.0
2,e_build_flag,0,2,2,0,0.000,1.0
3,e_build_noise,0,3,3,0,0.000,1.0


Read that table carefully. Rows with `masked=1` have no use signals but still
carry the turn's `outcome_score`. Those rows are the counterfactual — they are
half of what Layer 2 estimates from.

In [5]:
run.close()

## 3. The mock run

240 turns, zero tokens. This is where you debug.

In [6]:
cfg = Config(run_id="mock_240", backend="mock", mask_p=0.15,
             max_turns=240, db_path="data/merit.db")
run = Layer1Runner(cfg, client=MockClient(competence=0.9, seed=cfg.seed))
results = run.run()
df = run.ledger.dataframe(cfg.run_id)
len(df)

  turn  10  mean outcome 0.80  empty 0  tokens 1,573
  turn  20  mean outcome 0.80  empty 0  tokens 3,014
  turn  30  mean outcome 0.73  empty 0  tokens 4,590
  turn  40  mean outcome 0.70  empty 0  tokens 6,193
  turn  50  mean outcome 0.68  empty 0  tokens 7,720
  turn  60  mean outcome 0.72  empty 0  tokens 9,323
  turn  70  mean outcome 0.71  empty 0  tokens 10,856
  turn  80  mean outcome 0.70  empty 0  tokens 12,457
  turn  90  mean outcome 0.72  empty 0  tokens 13,970
  turn 100  mean outcome 0.69  empty 0  tokens 15,547
  turn 110  mean outcome 0.70  empty 0  tokens 17,095
  turn 120  mean outcome 0.70  empty 0  tokens 18,648
  turn 130  mean outcome 0.71  empty 0  tokens 20,235
  turn 140  mean outcome 0.71  empty 0  tokens 21,742
  turn 150  mean outcome 0.71  empty 0  tokens 23,405
  turn 160  mean outcome 0.72  empty 0  tokens 25,015
  turn 170  mean outcome 0.72  empty 0  tokens 26,657
  turn 180  mean outcome 0.72  empty 0  tokens 28,195
  turn 190  mean outcome 0.73  emp

960

## 4. Phase 0 check — does it pass?

### 4a. Ledger integrity

In [7]:
V.integrity(df, cfg.mask_p)

,check,pass,detail
0,all rows settled,True,0 open
1,one outcome per turn,True,0 turns disagree
2,mask rate near p,True,0.155 vs p=0.15
3,masked rows exist,True,149 rows
4,no use on masked rows,True,0 leaked
5,few empty completions,True,0 rows (0.0%)


### 4b. Detector sanity

`behavioural_rate` should be clearly positive for `useful` and ~0 for the rest.
`lexical_mean` stays low against the mock because the mock's replies are one
line long; against a real model it carries more.

In [8]:
V.detector_report(df, store)

,planted,injections,behavioural_rate,lexical_mean,semantic_mean
0,distractor,360,0.000000,0.00000,0.0
1,irrelevant,53,0.000000,0.00000,0.0
2,useful,398,0.502513,0.06511,0.0


### 4c. Signal check

Naive difference in means — no controls, no confidence intervals. Layer 2
replaces this properly. Its only job here is to show the effect exists.

Ignore the `irrelevant` rows: they are rarely retrieved, so `n_absent` is tiny
and their numbers are noise.

In [9]:
effects = V.naive_effect(df, store)
effects

,entry_id,planted,n_present,n_absent,n_strata,naive_effect,spread
0,e_deploy_flag,useful,60,10,1,0.750,0.000
1,e_misc_4,irrelevant,9,1,1,0.667,0.000
2,e_test_flag,useful,166,34,3,0.362,0.471
3,e_build_flag,useful,172,28,3,0.260,0.713
4,e_build_noise,distractor,135,25,3,0.120,0.263
5,e_misc_1,irrelevant,26,4,2,0.106,0.174
6,e_test_noise,distractor,144,26,3,0.082,0.251
7,e_misc_2,irrelevant,9,1,1,-0.111,0.000
8,e_misc_3,irrelevant,9,1,1,-0.111,0.000
9,e_deploy_noise,distractor,81,19,2,-0.272,0.038


In [10]:
print(V.verdict(effects))

PASS: every useful entry separates from every distractor. The instrument can see the effect. Proceed to Layer 2.


## 5. Sensitivity: how many turns do you need?

Re-run the check on growing prefixes of the ledger. The point where `useful`
separates cleanly from `distractor` is your minimum sample size for the real
run — report this number in the paper.

In [11]:
rows = []
for n in (40, 80, 120, 160, 200, 240):
    turn_ids = df["turn_id"].drop_duplicates().head(n)
    sub = df[df["turn_id"].isin(turn_ids)]
    e = V.naive_effect(sub, store)
    u = e[e.planted == "useful"]["naive_effect"].dropna()
    d = e[e.planted == "distractor"]["naive_effect"].dropna()
    rows.append({"turns": n,
                 "useful_mean": round(u.mean(), 3) if len(u) else None,
                 "distractor_mean": round(d.mean(), 3) if len(d) else None,
                 "gap": round(u.mean() - d.mean(), 3) if len(u) and len(d) else None,
                 "separated": bool(len(u) and len(d) and u.min() > d.max())})
pd.DataFrame(rows)

,turns,useful_mean,distractor_mean,gap,separated
0,40,0.512,-0.157,0.669,False
1,80,0.491,-0.101,0.592,False
2,120,0.475,-0.004,0.479,False
3,160,0.482,-0.046,0.528,True
4,200,0.485,0.006,0.479,True
5,240,0.457,-0.023,0.481,True


## 6. Masking rate sweep

`p` is the exploration tax. Too low and entries never get an absent row; too
high and you degrade real turns. Find the knee here, not on live API calls.

In [12]:
sweep = []
for p in (0.05, 0.10, 0.15, 0.25, 0.40):
    c = Config(run_id=f"p{p}", backend="mock", mask_p=p, max_turns=240,
               db_path="data/sweep.db")
    r = Layer1Runner(c, client=MockClient(competence=0.9, seed=c.seed))
    r.run(verbose=False)
    d = r.ledger.dataframe(c.run_id)
    e = V.naive_effect(d, store)
    u = e[e.planted == "useful"]["naive_effect"].dropna()
    dd = e[e.planted == "distractor"]["naive_effect"].dropna()
    sweep.append({"p": p,
                "mean_outcome": round(d.groupby("turn_id")["outcome_score"].first().mean(), 3),
                "gap": round(u.mean() - dd.mean(), 3) if len(u) and len(dd) else None,
                "min_useful": round(u.min(), 3) if len(u) else None,
                "max_distractor": round(dd.max(), 3) if len(dd) else None})
    r.close()
pd.DataFrame(sweep)

,p,mean_outcome,gap,min_useful,max_distractor
0,0.05,0.783,0.436,0.387,0.173
1,0.10,0.754,0.408,0.219,0.235
2,0.15,0.692,0.481,0.260,0.120
3,0.25,0.571,0.575,0.311,0.031
4,0.40,0.442,0.510,0.387,0.040


`mean_outcome` is the cost of masking — the performance you gave up to buy the
counterfactual. `gap` is what you bought. Report both.

In [13]:
c = Config(run_id="floor", backend="groq", model="openai/gpt-oss-20b",
           mask_p=1.0, max_turns=24, max_tokens=1024, max_total_tokens=10_000,
           db_path="data/merit.db")
with Layer1Runner(c) as r:
    r.run()
    t = r.ledger.turns_dataframe("floor")
    print("guess rate:", t["outcome_score"].mean())
    print("empty:", (t["error"] == "empty_completion").sum(), "/", len(t))

  turn  10  mean outcome 0.00  empty 3  tokens 7,660
[budget] stopping at turn 15: 10378 tokens spent (cap 10000)
done: 15 turns, 10,378 tokens, 0 unsettled rows, 3 empty completions
guess rate: 0.0
empty: 3 / 15


## 7. Live smoke test — spends tokens

Everything above was free. Run this **once**, after the mock checks pass.

Put your keys in `.env` at the repo root:

```
GROQ_API_KEY_1=gsk_...
GROQ_API_KEY_2=gsk_...
GROQ_API_KEY_3=gsk_...
GROQ_API_KEY_4=gsk_...
```

In [14]:
from dotenv import load_dotenv
load_dotenv("../.env")
from merit.clients import GroqKeyPool
pool = GroqKeyPool()
print(f"{len(pool.states)} keys loaded")

4 keys loaded


In [15]:
run.close()   # release the mock run's handle first

live = Config(run_id="live_400", backend="groq", model="openai/gpt-oss-20b",
              mask_p=0.15, max_turns=400, max_total_tokens=200_000,
              db_path="data/merit.db")
live_run = Layer1Runner(live)
live_results = live_run.run()
ldf = live_run.ledger.dataframe("live_400")
print(live_run.client.pool.report())

  turn  10  mean outcome 0.80  empty 0  tokens 4,440
  turn  20  mean outcome 0.85  empty 0  tokens 8,552
  turn  30  mean outcome 0.86  empty 2  tokens 14,181
  turn  40  mean outcome 0.87  empty 2  tokens 18,046
  turn  50  mean outcome 0.83  empty 2  tokens 22,916
  turn  60  mean outcome 0.86  empty 2  tokens 26,575
  turn  70  mean outcome 0.86  empty 4  tokens 31,507
  turn  80  mean outcome 0.87  empty 5  tokens 36,600
  turn  90  mean outcome 0.88  empty 5  tokens 40,489
  turn 100  mean outcome 0.87  empty 5  tokens 45,160
  turn 110  mean outcome 0.88  empty 6  tokens 49,699
  turn 120  mean outcome 0.87  empty 6  tokens 54,080
  turn 130  mean outcome 0.88  empty 6  tokens 58,050
  turn 140  mean outcome 0.87  empty 6  tokens 62,056
  turn 150  mean outcome 0.88  empty 6  tokens 66,624
  turn 160  mean outcome 0.88  empty 6  tokens 70,243
  turn 170  mean outcome 0.88  empty 6  tokens 75,134
  turn 180  mean outcome 0.87  empty 6  tokens 79,301
  turn 190  mean outcome 0.88 

In [16]:
display(V.integrity(ldf, live.mask_p))
display(V.detector_report(ldf, store))
le = V.naive_effect(ldf, store)
display(le)
print(V.verdict(le))

,check,pass,detail
0,all rows settled,True,0 open
1,one outcome per turn,True,0 turns disagree
2,mask rate near p,True,0.155 vs p=0.15
3,masked rows exist,True,248 rows
4,no use on masked rows,True,0 leaked
5,few empty completions,True,44 rows (2.8%)


,planted,injections,behavioural_rate,lexical_mean,semantic_mean
0,distractor,608,0.000000,0.055367,0.0
1,irrelevant,84,0.000000,0.000000,0.0
2,useful,660,0.536364,0.142595,0.0


,entry_id,planted,n_present,n_absent,n_strata,naive_effect,spread
0,e_deploy_flag,useful,94,16,1,0.989,0.000
1,e_misc_4,irrelevant,15,1,1,0.867,0.000
2,e_test_flag,useful,273,53,3,0.451,0.468
3,e_build_flag,useful,284,41,3,0.381,0.581
4,e_build_noise,distractor,222,42,3,0.171,0.030
5,e_test_noise,distractor,240,37,3,0.015,0.052
6,e_deploy_noise,distractor,127,29,2,-0.026,0.016
7,e_misc_2,irrelevant,16,1,1,-0.062,0.000
8,e_misc_3,irrelevant,15,2,1,-0.067,0.000
9,e_misc_1,irrelevant,37,11,2,-0.154,0.115


PASS: every useful entry separates from every distractor. The instrument can see the effect. Proceed to Layer 2.


In [17]:
live_run.ledger.turns_dataframe("live_60")[
    ["task_id","n_injected","outcome_score","turn_tokens","response_text"]
].head(12)

,task_id,n_injected,outcome_score,turn_tokens,response_text
0,t_test_00,4,1.0,558,dialyx test --runner-profile=ci7
1,t_test_01,4,1.0,450,dialyx run --runner-profile=ci7
2,t_test_02,3,1.0,346,dialyx test --runner-profile=ci7
3,t_test_03,3,0.0,513,--quiet
4,t_test_04,3,1.0,458,Run the runner with the option `--runner-profi...
5,t_test_05,3,1.0,593,dialyx run --runner-profile=ci7
6,t_test_06,3,0.0,481,Run the tests with the `--quiet` flag.
7,t_test_07,4,1.0,319,--runner-profile=ci7
8,t_build_00,4,1.0,441,Use the `--arch-profile=a9` flag when building...
9,t_build_01,3,1.0,301,Pass `--arch-profile=a9`.


Read a dozen real responses by hand. You are looking for the detector
disagreeing with your own judgement — a turn where the model clearly used the
note but `used_behavioural` is 0, or the reverse. Every such case is a detector
bug, and detector bugs become Layer 2 noise.

In [20]:
from pathlib import Path
from merit import Ledger

DB  = "data/merit.db"                 # same path the runner used
OUT = Path("../data"); OUT.mkdir(exist_ok=True)

lg = Ledger(DB)
for run in ("live_400", "live_60", "floor"):
    led, trn = lg.dataframe(run), lg.turns_dataframe(run)
    if led.empty:
        print(f"{run}: not in this db, skipped"); continue
    led.to_csv(OUT / f"{run}_ledger.csv", index=False)
    trn.to_csv(OUT / f"{run}_turns.csv", index=False)
    print(f"{run}: {len(led)} ledger rows, {len(trn)} turns")
(OUT / "live_400_config.json").write_text(live.to_json())
lg.close()

live_400: 1600 ledger rows, 400 turns
live_60: 240 ledger rows, 60 turns
floor: 60 ledger rows, 15 turns


In [19]:
live_run.close()